In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.nn.utils.rnn import pad_sequence
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

os.makedirs('../data/images/bert_bertweet_large', exist_ok=True)
os.makedirs('../models/bert_bertweet_large', exist_ok=True)

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================
#
# vinai/bertweet-large: the large variant of BERTweet.
# Same domain-specific pre-training on Twitter data as bertweet-base,
# but with a larger architecture (24 layers vs 12, hidden size 1024 vs 768).
#
# Comparison:
#   bertweet-base  : 135M params, 12 layers, hidden 768,  max_length 128
#   bertweet-large : 355M params, 24 layers, hidden 1024, max_length 512
#
# Both are pre-trained on English tweets — same domain as this task.
# The large variant has more capacity to capture nuanced semantics.
# Model: https://huggingface.co/vinai/bertweet-large
#
# Label Smoothing (Szegedy et al. 2016):
# Instead of training with hard targets {0, 1}, label smoothing uses
# soft targets {ε/2, 1-ε/2} — e.g. {0.05, 0.95} with ε=0.1.
# This prevents the model from becoming overconfident on ambiguous samples,
# which is especially relevant here given the 18 mislabeled samples we
# corrected manually. Even after correction, some tweets remain genuinely
# ambiguous (e.g. metaphorical use of disaster keywords).
# Source: inspired by the multi-sample dropout notebook analysis.

MODEL_CHECKPOINT      = 'vinai/bertweet-large'
MAX_LENGTH            = 128   # tweets are short — 128 is sufficient even for large
BATCH_SIZE            = 8     # reduced from 16 due to larger model memory footprint
LEARNING_RATE         = 1e-5  # lower lr for large models — more stable convergence
NUM_EPOCHS            = 3
VAL_SIZE              = 0.1
SEED                  = 42
WARMUP_RATIO          = 0.1   # warm up for first 10% of steps
LABEL_SMOOTHING       = 0.1   # soft targets: {0.05, 0.95} instead of {0, 1}

In [ ]:
# =============================================================================
# LOAD DATA
# =============================================================================

df_full = pd.read_csv('../data/augmented_train.csv')
df_test = pd.read_csv('../data/test_cleaned.csv')

# Keep only original rows — augmented rows (back-translations) are excluded
# from validation to avoid inflated scores
original_max_id = df_full['id'].max() // 2
df_orig = df_full[df_full['id'] <= original_max_id].copy()

print(f"Original training rows : {len(df_orig)}")
print(f"Test rows              : {len(df_test)}")
print(f"\nLabel distribution (original):")
print(df_orig['target_relabeled'].value_counts(normalize=True).round(3))

In [ ]:
# =============================================================================
# TRAIN / VALIDATION SPLIT
# =============================================================================

df_work = df_orig[['text_cleaned', 'target_relabeled']].dropna()
df_work = df_work.rename(columns={'target_relabeled': 'label'})
df_work = df_work.drop_duplicates(subset='text_cleaned')

train_df, val_df = train_test_split(
    df_work,
    test_size=VAL_SIZE,
    stratify=df_work['label'],
    random_state=SEED
)

overlap = len(set(train_df['text_cleaned']) & set(val_df['text_cleaned']))
print(f"\nTrain size : {len(train_df)}")
print(f"Val size   : {len(val_df)}")
print(f"Overlap    : {overlap}  ← must be 0")

In [ ]:
# =============================================================================
# TOKENIZATION
# =============================================================================
# BERTweet-large uses BPE tokenization (same as RoBERTa) with a Twitter-
# specific vocabulary. normalization=True activates BERTweet's built-in
# tweet normalization: URLs → HTTPURL, mentions → @USER.
# use_fast=False is required for BERTweet tokenizer compatibility.

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_CHECKPOINT,
    use_fast=False,
    normalization=True
)

def tokenize(example):
    return tokenizer(
        example['text_cleaned'],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset   = Dataset.from_pandas(val_df.reset_index(drop=True))

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset   = val_dataset.map(tokenize, batched=True)

print(f"\nTokenization complete.")
print(f"Train dataset features: {train_dataset.features}")

In [ ]:
# =============================================================================
# MODEL
# =============================================================================
# vinai/bertweet-large: RoBERTa-large architecture, 24 Transformer layers,
# hidden size 1024, 16 attention heads, ~355M parameters.
# Pre-trained on 873M English tweets (Nguyen et al. 2020).
#
# ignore_mismatched_sizes=True is required because the pre-training checkpoint
# has a language model head with different output dimensions than the
# classification head we add here.
#
# UNEXPECTED keys = pre-training heads (not needed for classification)
# MISSING keys    = new classification head (randomly initialized, will be trained)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2,
    ignore_mismatched_sizes=True
)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n{'='*50}")
print(f"  PARAMETER COUNT")
print(f"{'='*50}")
print(f"  Total parameters     : {total_params:,}")
print(f"  Trainable parameters : {trainable_params:,}")
print(f"  Trainable %          : {100 * trainable_params / total_params:.2f}%")
print(f"{'='*50}")
print(f"\n  bertweet-base  : ~135M parameters")
print(f"  bertweet-large : ~355M parameters  ← this model")

In [ ]:
# =============================================================================
# METRICS
# =============================================================================

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {'f1': f1_score(labels, preds)}

In [ ]:
# =============================================================================
# TRAINING
# =============================================================================
# Key differences vs bert_bertweet.ipynb:
#   - label_smoothing_factor=0.1: soft targets prevent overconfidence on
#     ambiguous tweets (metaphorical disaster language, mislabeled samples)
#   - learning_rate=1e-5: lower than bertweet-base (2e-5) — large models
#     need more conservative updates to avoid catastrophic forgetting
#   - per_device_train_batch_size=8: reduced from 16 to fit GPU memory
#   - warmup_ratio=0.1: warm up LR for first 10% of steps (same as base)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir                  = '../models/bert_bertweet_large',
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    learning_rate               = LEARNING_RATE,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    num_train_epochs            = NUM_EPOCHS,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1',
    greater_is_better           = True,
    seed                        = SEED,
    warmup_ratio                = WARMUP_RATIO,
    label_smoothing_factor      = LABEL_SMOOTHING,  # soft targets {0.05, 0.95}
    report_to                   = 'none'
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    data_collator   = data_collator,
    compute_metrics = compute_metrics
)

print("\nStarting BERTweet-large fine-tuning...")
print(f"  Label smoothing : {LABEL_SMOOTHING}")
print(f"  Learning rate   : {LEARNING_RATE}")
print(f"  Batch size      : {BATCH_SIZE}")
print(f"  Warmup ratio    : {WARMUP_RATIO}")
train_result = trainer.train()
print("\nTraining complete.")
print(train_result)

In [ ]:
# =============================================================================
# EVALUATION — manual loop
# Works on both CPU (Windows deadlock workaround) and GPU (Lightning/Kaggle)
# =============================================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)
model.eval()
all_preds = []

for i in range(0, len(val_dataset), BATCH_SIZE):
    batch          = val_dataset[i:i+BATCH_SIZE]
    input_ids      = [torch.tensor(x) for x in batch['input_ids']]
    attention_mask = [torch.tensor(x) for x in batch['attention_mask']]
    input_ids      = pad_sequence(input_ids, batch_first=True, padding_value=0).to(device)
    attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0).to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

    preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
    all_preds.extend(preds)
    print(f"  {min(i+BATCH_SIZE, len(val_dataset))}/{len(val_dataset)}")

val_preds  = np.array(all_preds)
val_labels = val_df['label'].values
val_f1     = f1_score(val_labels, val_preds)

print(f"\nValidation F1 : {val_f1:.4f}")
print("\nDetailed Classification Report:")
print(classification_report(val_labels, val_preds,
      target_names=['Not Disaster', 'Disaster']))

eval_result = {'eval_f1': val_f1}

In [ ]:
# =============================================================================
# TRAINING CURVES
# =============================================================================

epochs      = []
val_loss    = []
val_f1_hist = []

for entry in trainer.state.log_history:
    if 'eval_loss' in entry:
        epochs.append(entry['epoch'])
        val_loss.append(entry['eval_loss'])
        val_f1_hist.append(entry.get('eval_f1', None))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, val_loss, marker='o', label='val_loss', color='#d62728')
axes[0].set_title('Validation Loss per Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(epochs, val_f1_hist, marker='o', label='val_f1', color='#1f77b4')
axes[1].set_title('Validation F1 per Epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('F1')
axes[1].set_ylim(0.75, 0.92)
axes[1].legend()

plt.suptitle('BERTweet-large + Label Smoothing — Training Curves',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/images/bert_bertweet_large/bertweet_large_training_curves.png',
            bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# ERROR ANALYSIS
# =============================================================================

fp = val_df[(val_preds == 1) & (val_labels == 0)]['text_cleaned'].head(5)
fn = val_df[(val_preds == 0) & (val_labels == 1)]['text_cleaned'].head(5)

print("False Positives (predicted disaster, actually not):")
print(fp.to_string())
print("\nFalse Negatives (predicted not disaster, actually is):")
print(fn.to_string())

In [ ]:
# =============================================================================
# FULL MODEL COMPARISON
# =============================================================================

print(f"\n{'='*60}")
print(f"  FULL MODEL COMPARISON")
print(f"{'='*60}")
print(f"  {'Model':<40} {'Val F1':<10} {'Kaggle F1'}")
print(f"  {'-'*60}")
print(f"  {'BiLSTM + GloVe 100d':<40} {'0.776':<10} {'0.809'}")
print(f"  {'BERT fine-tuning (bert-base-uncased)':<40} {'0.838':<10} {'0.839'}")
print(f"  {'BERT + LoRA (r=8)':<40} {'0.827':<10} {'0.833'}")
print(f"  {'BERTweet fine-tuning (bertweet-base)':<40} {'0.821':<10} {'0.839'}")
print(f"  {'BERTweet + LoRA (r=8)':<40} {'0.813':<10} {'0.846'}")
print(f"  {'BERTweet-large + label smoothing':<40} {val_f1:.3f:<10} {'(submit)'}")
print(f"{'='*60}")

In [ ]:
# =============================================================================
# GENERATE SUBMISSION FILE
# =============================================================================

test_work    = df_test[['text_cleaned']].copy()
test_dataset = Dataset.from_pandas(test_work.reset_index(drop=True))
test_dataset = test_dataset.map(tokenize, batched=True)

model.eval()
all_test_preds = []

for i in range(0, len(test_dataset), BATCH_SIZE):
    batch          = test_dataset[i:i+BATCH_SIZE]
    input_ids      = [torch.tensor(x) for x in batch['input_ids']]
    attention_mask = [torch.tensor(x) for x in batch['attention_mask']]
    input_ids      = pad_sequence(input_ids, batch_first=True, padding_value=0)
    attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

    preds = torch.argmax(outputs.logits, dim=1).numpy()
    all_test_preds.extend(preds)
    print(f"  {min(i+BATCH_SIZE, len(test_dataset))}/{len(test_dataset)}")

submission = pd.DataFrame({
    'id'    : df_test['id'],
    'target': np.array(all_test_preds)
})
submission.to_csv('../data/submission_bertweet_large.csv', index=False)

print(f"\nSubmission saved to ../data/submission_bertweet_large.csv")
print(submission['target'].value_counts())

In [ ]:
# =============================================================================
# KAGGLE SUBMISSION RESULTS
# =============================================================================
# Upload ../data/submission_bertweet_large.csv to:
# https://www.kaggle.com/competitions/nlp-getting-started/submissions

print(f"\n{'='*50}")
print(f"  KAGGLE SUBMISSION RESULTS")
print(f"{'='*50}")
print(f"  Model                   : vinai/bertweet-large")
print(f"  Label smoothing         : {LABEL_SMOOTHING}")
print(f"  Epochs                  : {NUM_EPOCHS}")
print(f"  Learning rate           : {LEARNING_RATE}")
print(f"  Batch size              : {BATCH_SIZE}")
print(f"  Warmup ratio            : {WARMUP_RATIO}")
print(f"  Val F1                  : {eval_result['eval_f1']:.4f}")
print(f"  Kaggle Public F1        : (submit to update)")
print(f"  Submission file         : ../data/submission_bertweet_large.csv")
print(f"  Competition link        : https://www.kaggle.com/competitions/nlp-getting-started/submissions")
print(f"{'='*50}")